# Directors (Companies House) + managing director (company website only)

Reads **`Market Research List.csv`**, resolves **Company House Number**, then:

1. Fetches **active** directors from the **Companies House API** (separate from managing director).
2. Copies **Company Email** from the list into **Company email (from list)**.
3. **Managing director** is inferred **only from the company website**: it fetches common pages on **Company URL** (homepage, about, team, etc.) and parses visible text; if nothing is found, it runs a **DuckDuckGo** query restricted to that site (`site:yourdomain`). **It does not use Companies House** for the managing director field.

**Requirements:** `Company URL` (or `Company url` / `Website`) on each row for MD lookup; otherwise managing director is left empty. **`pandas`**, **`requests`**, **`beautifulsoup4`**, **`COMPANIES_HOUSE_API_KEY`** (or **`.env`**). Toggles: **`WEBSITE_MANAGING_DIRECTOR`**, **`WEBSITE_MD_SITE_SEARCH`** (DDG `site:` fallback), delays below.

In [1]:
import os
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests

def _load_dotenv(path: str = ".env") -> None:
    p = Path(path)
    if not p.is_file():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if k and not os.environ.get(k):
            os.environ[k] = v


_load_dotenv()

BASE_URL = os.environ.get(
    "COMPANIES_HOUSE_BASE_URL", "https://api.companieshouse.gov.uk"
).strip().rstrip("/")
HEADERS = {"Accept": "application/json"}
HTTP_UA = {"User-Agent": "Mozilla/5.0 (compatible; SeaBrightInsight/1.2; +research)"}

INPUT_CSV = "Market Research List.csv"
OUTPUT_CSV = "market_research_directors.csv"

MAX_COMPANIES = None

WEBSITE_MANAGING_DIRECTOR = True
WEBSITE_MD_SITE_SEARCH = True
WEB_SCRAPE_DELAY_SEC = 0.35
WEB_SEARCH_DELAY_SEC = 1.5
WEB_REQUEST_TIMEOUT = 14

api_key = os.environ.get("COMPANIES_HOUSE_API_KEY", "").strip()
if not api_key:
    api_key = input("Companies House API key: ").strip()
if not api_key:
    raise ValueError("Set COMPANIES_HOUSE_API_KEY or enter a key when prompted.")

auth = requests.auth.HTTPBasicAuth(api_key, "")

In [2]:
from bs4 import BeautifulSoup


def ch_name_to_full_display(name: str) -> str:
    name = (name or "").strip()
    if "," in name:
        last, first = [p.strip() for p in name.split(",", 1)]
        if first and last:
            return f"{first} {last}".strip()
    return name


def is_active_officer(item: Dict[str, Any]) -> bool:
    return not item.get("resigned_on")


def is_director_like(item: Dict[str, Any]) -> bool:
    role = (item.get("officer_role") or "").lower().replace(" ", "").replace("-", "")
    return "director" in role or "designatedmember" in role


def fetch_officers_page(
    company_number: str, start_index: int = 0, items_per_page: int = 100
) -> Optional[Dict[str, Any]]:
    url = f"{BASE_URL}/company/{company_number}/officers"
    params = {"start_index": start_index, "items_per_page": items_per_page}
    r = requests.get(url, auth=auth, headers=HEADERS, params=params, timeout=40)
    if r.status_code == 200:
        return r.json()
    return None


def fetch_all_officers(company_number: str) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    start = 0
    page_size = 100
    while True:
        data = fetch_officers_page(company_number, start_index=start, items_per_page=page_size)
        if not data:
            break
        items = data.get("items") or []
        out.extend(items)
        total = int(data.get("total_results") or len(out))
        start += len(items)
        if start >= total or not items:
            break
    return out


def active_directors(company_number: str) -> List[Dict[str, str]]:
    rows = []
    for item in fetch_all_officers(company_number):
        if not is_active_officer(item) or not is_director_like(item):
            continue
        raw_name = (item.get("name") or "").strip()
        if not raw_name:
            continue
        rows.append(
            {
                "officer_role": (item.get("officer_role") or "").strip(),
                "occupation": (item.get("occupation") or "").strip(),
                "name_ch": raw_name,
                "name_full": ch_name_to_full_display(raw_name),
                "appointed_on": (item.get("appointed_on") or "").strip(),
            }
        )
    return rows


def site_origin(website_url: str) -> str:
    u = (website_url or "").strip()
    if not u:
        return ""
    if not u.startswith(("http://", "https://")):
        u = "https://" + u
    try:
        p = urlparse(u)
        if not p.scheme or not p.netloc:
            return ""
        return f"{p.scheme}://{p.netloc}"
    except ValueError:
        return ""


def site_host(website_url: str) -> str:
    o = site_origin(website_url)
    if not o:
        return ""
    h = (urlparse(o).hostname or "").lower()
    return h[4:] if h.startswith("www.") else h


def fetch_page_html(url: str) -> str:
    try:
        r = requests.get(
            url,
            headers=HTTP_UA,
            timeout=WEB_REQUEST_TIMEOUT,
            allow_redirects=True,
        )
        if r.status_code == 200 and r.text:
            ct = (r.headers.get("Content-Type") or "").lower()
            if "html" in ct or not ct:
                return r.text
    except requests.RequestException:
        pass
    return ""


def html_to_visible_text(html: str, max_chars: int = 80000) -> str:
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    t = soup.get_text(" ", strip=True)
    t = re.sub(r"\s+", " ", t).strip()
    return t[:max_chars]


_MD_NAME = re.compile(
    r"\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\b"
)

_MD_PATTERNS = [
    re.compile(
        r"(?:Managing\s+Director|M\.?D\.?)\s*[\u2013\u2014:\-]\s*([A-Z][^\n\|<,]{2,55})",
        re.I,
    ),
    re.compile(
        r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\s*,\s*(?:Managing\s+Director|MD)\b",
        re.I,
    ),
    re.compile(
        r"(?:led by|run by|headed by)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\b",
        re.I,
    ),
]


def _clean_md_candidate(s: str) -> str:
    s = re.sub(r"\s+", " ", (s or "").strip())
    s = re.sub(r"[|\[\]].*", "", s).strip()
    for junk in (
        "LinkedIn",
        "Facebook",
        "Twitter",
        "Wikipedia",
        "Cookie",
        "Read more",
        "Click here",
    ):
        if junk.lower() in s.lower():
            return ""
    m = _MD_NAME.search(s)
    return m.group(1).strip() if m else s[:80].strip()


def extract_managing_director_from_plain_text(blob: str) -> str:
    if not blob:
        return ""
    blob = blob[:80000]
    low = blob.lower()
    for pat in _MD_PATTERNS:
        m = pat.search(blob)
        if m:
            hit = _clean_md_candidate(m.group(1))
            if hit and len(hit) > 3:
                return hit
    for needle in ("managing director", "chief executive officer", "chief executive"):
        idx = low.find(needle)
        if idx >= 0:
            window = blob[max(0, idx - 140) : idx + 200]
            m = _MD_NAME.search(window)
            if m:
                hit = _clean_md_candidate(m.group(1))
                if hit:
                    return hit
    return ""


def ddg_html_search(query: str) -> str:
    try:
        r = requests.post(
            "https://html.duckduckgo.com/html/",
            data={"q": query, "b": "", "kl": "uk-en"},
            headers=HTTP_UA,
            timeout=WEB_REQUEST_TIMEOUT,
        )
        if r.ok and r.text:
            return r.text
    except requests.RequestException:
        pass
    return ""


def extract_managing_director_from_search_snippets(html: str) -> str:
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    parts: List[str] = []
    for a in soup.find_all("a", class_=re.compile("result__a", re.I)):
        parts.append(a.get_text(" ", strip=True))
    for cls in ("result__snippet", "web-result__description", "result__body"):
        for el in soup.find_all(class_=re.compile(cls, re.I)):
            parts.append(el.get_text(" ", strip=True))
    blob = "\n".join(parts) if parts else soup.get_text(" ", strip=True)[:12000]
    return extract_managing_director_from_plain_text(blob)


_SCRAPE_PATHS = [
    "/",
    "/about",
    "/about-us",
    "/our-story",
    "/who-we-are",
    "/team",
    "/our-team",
    "/leadership",
    "/people",
    "/meet-the-team",
    "/contact",
    "/contact-us",
]


def managing_director_from_company_website(website_url: str) -> Tuple[str, str]:
    """Managing director from company site only (scrape + optional site-limited search). Not Companies House."""
    origin = site_origin(website_url)
    if not origin:
        return "", "no-website-url"
    host = site_host(website_url)
    if not host:
        return "", "no-website-url"

    seen: set[str] = set()
    for path in _SCRAPE_PATHS:
        page_url = urljoin(origin.rstrip("/") + "/", path.lstrip("/"))
        if page_url in seen:
            continue
        seen.add(page_url)
        html = fetch_page_html(page_url)
        if not html:
            continue
        text = html_to_visible_text(html)
        hit = extract_managing_director_from_plain_text(text)
        if hit:
            return hit, "website-scrape"
        time.sleep(WEB_SCRAPE_DELAY_SEC)

    if WEBSITE_MD_SITE_SEARCH:
        q = f'managing director site:{host}'
        html = ddg_html_search(q)
        time.sleep(WEB_SEARCH_DELAY_SEC)
        hit = extract_managing_director_from_search_snippets(html)
        if hit:
            return hit, "website-site-search"

    return "", "none"


def normalize_company_number(raw: object, url_fallback: object = "") -> str:
    if url_fallback is None or pd.isna(url_fallback):
        u = ""
    else:
        u = str(url_fallback).strip()
    m = re.search(r"/company/([^/\s?]+)", u, re.I)
    if m:
        return m.group(1).strip().upper()
    if pd.isna(raw):
        return ""
    s = str(raw).strip()
    if re.match(r"^n/a", s, re.I) or s.lower() in ("nan", "n/a", "na", "-", ""):
        s = ""
    s = re.sub(r"\.0$", "", s).strip()
    m = re.search(r"/company/([^/\s?]+)", s, re.I)
    if m:
        return m.group(1).strip().upper()
    compact = re.sub(r"\s+", "", s)
    if re.fullmatch(r"\d{1,8}", compact):
        return compact.zfill(8)
    mu = compact.upper()
    m2 = re.fullmatch(r"([A-Z]{1,2}\d{5,7})", mu)
    if m2:
        return m2.group(1)
    m3 = re.search(r"\b([A-Z]{1,2}\d{5,7})\b", mu)
    if m3:
        return m3.group(1)
    m4 = re.search(r"\d{6,8}", compact)
    if m4:
        return m4.group(0).zfill(8)
    return ""

In [3]:
src = pd.read_csv(INPUT_CSV)
col_lower = {c.lower(): c for c in src.columns}


def col(*names: str) -> str:
    for n in names:
        if n.lower() in col_lower:
            return col_lower[n.lower()]
    raise KeyError(f"None of columns {names} found in {list(src.columns)}")


c_name = col("Company Name", "company name")
c_num = col("Company House Number", "company number", "Company Number")
c_url = None
try:
    c_url = col("Company House URL", "company house url")
except KeyError:
    pass
c_email_list = None
try:
    c_email_list = col("Company Email", "company email")
except KeyError:
    pass
c_website = None
try:
    c_website = col("Company URL", "company url", "Website", "website")
except KeyError:
    pass

url_series = src[c_url].astype(str) if c_url else pd.Series([""] * len(src))
num_raw = src[c_num]
src["_num"] = [
    normalize_company_number(num_raw.iloc[i], url_series.iloc[i] if c_url else "")
    for i in range(len(src))
]
skipped = len(src) - (src["_num"].str.len() > 0).sum()
src = src[src["_num"].str.len() > 0].copy()
print(f"Rows with a usable company number: {len(src)} (skipped {skipped} without number)")
if c_website is None:
    print("Warning: no Company URL / Website column — Managing director will stay empty.")
src[[c_name, c_num]].head()

Rows with a usable company number: 47 (skipped 3 without number)


,Company Name,Company House Number
0,Kings Estates,11436373
1,Brighton & Hove museums,11774969
2,Brighton Pier Group,8687172
4,Nightcap Ltd,12899067
5,Drusilla's Park,3261226


In [ ]:
out_rows: List[Dict[str, str]] = []

_src = (
    src
    if MAX_COMPANIES is None
    else src.head(int(MAX_COMPANIES)).copy()
)
_total = len(_src)

for i, (_, row) in enumerate(_src.iterrows(), 1):
    company_name = str(row[c_name]).strip()
    num = str(row["_num"]).strip()
    ch_url = ""
    if c_url:
        ch_url = str(row.get(c_url, "") or "").strip()
    if not ch_url and num:
        ch_url = f"https://find-and-update.company-information.service.gov.uk/company/{num}"

    company_email_list = ""
    if c_email_list is not None:
        v = row.get(c_email_list, "")
        company_email_list = "" if pd.isna(v) else str(v).strip()

    website = ""
    if c_website is not None:
        wv = row.get(c_website, "")
        website = "" if pd.isna(wv) else str(wv).strip()

    print(f"[{i}/{_total}] {company_name} ({num}) ...", flush=True)
    dirs_ = active_directors(num)

    md_final, md_source = "", "disabled"
    if WEBSITE_MANAGING_DIRECTOR:
        if not website.strip():
            md_final, md_source = "", "no-website-url"
        else:
            md_final, md_source = managing_director_from_company_website(website)

    company_block: List[Dict[str, str]] = []
    if not dirs_:
        row_dict = {
            "Company Name": company_name,
            "Company House Number": num,
            "Company House URL": ch_url,
            "Company URL": website,
            "Company email (from list)": company_email_list,
            "Managing director": md_final,
            "Managing director source": md_source,
            "Director name (display)": "",
            "Director name (as CH)": "",
            "Officer role": "",
            "Officer occupation": "",
            "Appointed on": "",
        }
        out_rows.append(row_dict)
        company_block.append(row_dict)
    else:
        for d in dirs_:
            row_dict = {
                "Company Name": company_name,
                "Company House Number": num,
                "Company House URL": ch_url,
                "Company URL": website,
                "Company email (from list)": company_email_list,
                "Managing director": md_final,
                "Managing director source": md_source,
                "Director name (display)": d["name_full"],
                "Director name (as CH)": d["name_ch"],
                "Officer role": d["officer_role"],
                "Officer occupation": d["occupation"],
                "Appointed on": d["appointed_on"],
            }
            out_rows.append(row_dict)
            company_block.append(row_dict)

    print("-" * 72, flush=True)
    print(f"OUTCOME - {company_name}  |  CH: {num}", flush=True)
    if c_website is not None:
        print(f"  Company URL: {website or '(none)'}", flush=True)
    else:
        print("  Company URL: (column missing)", flush=True)
    print(f"  Company email (from list): {company_email_list or '(none)'}", flush=True)
    print(
        f"  Managing director (website only): {md_final or '(none)'}  "
        f"[source: {md_source}]",
        flush=True,
    )
    if not dirs_:
        print("  Directors (Companies House): (none)", flush=True)
    else:
        for j, r in enumerate(company_block, 1):
            print(
                f"  Director {j}: {r['Director name (display)'] or '(?)'}  "
                f"[as CH: {r['Director name (as CH)']} | {r['Officer role']} | "
                f"occ: {r.get('Officer occupation') or '-'} | appointed {r['Appointed on']}]",
                flush=True,
            )
    print("-" * 72, flush=True)

    time.sleep(0.35)

result = pd.DataFrame(out_rows)
result.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(result)} rows to {OUTPUT_CSV}")
result.head(20)

[1/47] Kings Estates (11436373) ...
------------------------------------------------------------------------
OUTCOME - Kings Estates  |  CH: 11436373
  Company URL: Kings Estates
  Company email (from list): hello@kings-estates.co.uk
  Managing director (website only): (none)  [source: none]
  Director 1: Andrew Mark KING  [as CH: KING, Andrew Mark | director | occ: - | appointed 2018-06-27]
  Director 2: Imelda KING  [as CH: KING, Imelda | director | occ: - | appointed 2018-06-27]
------------------------------------------------------------------------
[2/47] Brighton & Hove museums (11774969) ...
------------------------------------------------------------------------
OUTCOME - Brighton & Hove museums  |  CH: 11774969
  Company URL: https://brightonmuseums.org.uk/
  Company email (from list): info@rpmt.org.uk
  Managing director (website only): Head Gardener  [source: website-scrape]
  Director 1: James ALEXANDER  [as CH: ALEXANDER, James | director | occ: - | appointed 2025-05-01]
 